In [1]:
import pandas as pd
import zipfile

In [6]:
import pandas as pd

n = 50
df = pd.read_csv("../data/NQ/Natural-Questions-Filtered.csv", nrows=n)

for _, r in df.iterrows():
    print("Full Q",r)
    print(f"Q: {r['question']}\n  A: {r['short_answers']!r}\n Length: {len(r['short_answers'])!r}\n")



Full Q question         which is the most common use of opt-in e-mail ...
long_answers     A common example of permission marketing is a ...
short_answers    A newsletter sent to an advertising firm's cus...
Name: 0, dtype: str
Q: which is the most common use of opt-in e-mail marketing
  A: "A newsletter sent to an advertising firm's customers"
 Length: 52

Full Q question                   how i.met your mother who is the mother
long_answers     Tracy McConnell, better known as `` The Mother...
short_answers                                      Tracy McConnell
Name: 1, dtype: str
Q: how i.met your mother who is the mother
  A: 'Tracy McConnell'
 Length: 15

Full Q question                          who had the most wins in the nfl
long_answers     Active quarterback Tom Brady holds the records...
short_answers                                            Tom Brady
Name: 2, dtype: str
Q: who had the most wins in the nfl
  A: 'Tom Brady'
 Length: 9

Full Q question               who played

In [11]:
import csv

path = "../data/NQ/Natural-Questions-Filtered.csv"

counts = {"total": 0, "short_only": 0, "long_only": 0, "both": 0, "neither": 0}

with open(path, encoding="utf-8") as f:
    for row in csv.DictReader(f):          # sep="\t" -> add delimiter="\t"
        s = bool(row["short_answers"].strip())
        l = bool(row["long_answers"].strip())
        counts["total"] += 1
        counts["both" if s and l else "short_only" if s else "long_only" if l else "neither"] += 1

counts["any_short"] = counts["short_only"] + counts["both"]
counts["any_long"] = counts["long_only"] + counts["both"]
print(counts)

{'total': 86212, 'short_only': 0, 'long_only': 0, 'both': 86212, 'neither': 0, 'any_short': 86212, 'any_long': 86212}


In [13]:
import re
import pandas as pd

PATH = "../data/NQ/Natural-Questions-Filtered.csv"
SPLIT = re.compile(r",\s+(?=[A-Z0-9])")   # the join signature from caveat 19

df = pd.read_csv(PATH)
s = df["short_answers"].fillna("")
n = len(df)

has_comma  = s.str.contains(",", regex=False)
looks_join = s.apply(lambda a: bool(SPLIT.search(a)))
inner_only = has_comma & ~looks_join

print(f"rows                             : {n:,}")
print(f"short answers containing a comma : {has_comma.sum():,}  ({has_comma.mean():.1%})")
print(f"  matching the multi-span join   : {looks_join.sum():,}  ({looks_join.mean():.1%})")
print(f"  comma but NOT a join           : {inner_only.sum():,}  ({inner_only.mean():.1%})")

spans = s[looks_join].apply(lambda a: len(SPLIT.split(a)))
print(f"  spans per joined answer        : mean {spans.mean():.2f}  max {spans.max()}")

rows                             : 86,212
short answers containing a comma : 19,423  (22.5%)
  matching the multi-span join   : 18,463  (21.4%)
  comma but NOT a join           : 960  (1.1%)
  spans per joined answer        : mean 2.57  max 21


Manually check types of errors and fix as many as possible. (date format is one)


# Check corrupt data full wiki

In [9]:
fullwiki = pd.read_csv("data/wikiDump/psgs_w100.tsv", sep="\t")
fullwiki.head()

KeyboardInterrupt: 

In [ ]:
fullwiki_nulls = fullwiki[fullwiki["text"].notna() & fullwiki["text"].str.strip().ne("")]
fullwiki_nulls

In [3]:
import csv

TARGET = 1_000_000
SRC = "data/wikiDump/psgs_w100.tsv"
DST = "data/wikiDump/wiki_1M.tsv"

with open(SRC, encoding="utf-8", errors="replace") as fin, \
     open(DST, "w", encoding="utf-8", newline="") as fout:
    reader = csv.DictReader(fin, delimiter="\t")
    writer = csv.DictWriter(fout, fieldnames=reader.fieldnames, delimiter="\t")
    writer.writeheader()
    for i, row in enumerate(reader):
        if i >= TARGET:
            break
        writer.writerow(row)

print(f"Done — {min(i+1, TARGET):,} passages written to {DST}")

Done — 1,000,000 passages written to data/wikiDump/wiki_1M.tsv


In [5]:
wiki1mil = pd.read_csv("data/wikiDump/wiki_1M.tsv", sep="\t")

wiki1mil.head()
wiki1mil.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 6 columns):
 #   Column           Non-Null Count    Dtype
---  ------           --------------    -----
 0   id               1000000 non-null  int64
 1   text             998692 non-null   str  
 2   wikipedia_title  1000000 non-null  str  
 3   wikipedia_id     1000000 non-null  int64
 4   end_paragraph    1000000 non-null  int64
 5   end_character    1000000 non-null  int64
dtypes: int64(4), str(2)
memory usage: 45.8 MB


In [ ]:
df = wiki1mil[wiki1mil["text"].notna() & wiki1mil["text"].str.strip().ne("")]
df.to_csv("data/wikiDump/wiki_1M_clean.tsv", sep="\t", index=False)


In [8]:
df = wiki1mil[wiki1mil["text"].isna() | wiki1mil["text"].str.strip().eq("")]
df

,id,text,wikipedia_title,wikipedia_id,end_paragraph,end_character
8074,8074,NaN,Daniel Chodowiecki,43655,19,38
14031,14031,NaN,"Ashfield, Massachusetts",116762,37,37
14715,14715,NaN,"Lawrence, Massachusetts",116748,346,83
18124,18124,NaN,Fairlight (group),621092,38,69
20290,20290,NaN,Alexandru Marghiloman,1214891,15,205
...,...,...,...,...,...,...
997952,997952,NaN,2016 in Scandinavian music,49414843,219,53
998246,998246,NaN,Selina Giles,49415983,8,311
998382,998382,NaN,Mark Spragg,49416623,20,67
998691,998691,NaN,Gud (music producer),49416870,21,36


In [ ]:
nq = pd.read_csv("data/NQ/Natural-Questions-Filtered.csv", sep=",")

nq.head()

,question,long_answers,short_answers
0,which is the most common use of opt-in e-mail ...,A common example of permission marketing is a ...,A newsletter sent to an advertising firm's cus...
1,how i.met your mother who is the mother,"Tracy McConnell, better known as `` The Mother...",Tracy McConnell
2,who had the most wins in the nfl,Active quarterback Tom Brady holds the records...,Tom Brady
3,who played mantis guardians of the galaxy 2,Pom Klementieff (born May 1986) is a French ac...,Pom Klementieff
4,the nashville sound brought a polished and cos...,"In the early 1960s, the Nashville sound began ...",The use of lush string arrangements with a rea...


In [30]:
nq.shape
nq.columns
nq.info()

<class 'pandas.DataFrame'>
RangeIndex: 86212 entries, 0 to 86211
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   question       86212 non-null  str  
 1   long_answers   86212 non-null  str  
 2   short_answers  86212 non-null  str  
dtypes: str(3)
memory usage: 2.0 MB


In [4]:
hardware = pd.read_csv("results/hardware.csv", sep=",")

hardware.head()

,timestamp,disk_r_s,disk_w_s,disk_rkB_s,disk_wkB_s,disk_r_await_ms,disk_w_await_ms,disk_util_pct,gpu_mem_used_mb,gpu_mem_total_mb,gpu_mem_free_mb,gpu_util_pct
0,2026-04-29T01:51:36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,692.0,8192.0,7368.0,1.0
1,2026-04-29T01:51:37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,698.0,8192.0,7362.0,0.0
2,2026-04-29T01:51:39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,714.0,8192.0,7346.0,0.0
3,2026-04-29T01:51:40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,714.0,8192.0,7346.0,0.0
4,2026-04-29T01:51:42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,719.0,8192.0,7341.0,8.0
